# 04_build_vector_bm25_index

## 목적

이 노트북은 `03_build_parent_child_chunks.ipynb`에서 생성한 `children.jsonl`을 기반으로
구조화 RAG용 검색 인덱스를 생성한다.

기존 `vectordb_test.ipynb`와의 차이:

기존:
- PDF를 직접 로드
- RecursiveCharacterTextSplitter로 기본 chunking
- OpenAI text-embedding-3-small
- ChromaDB 저장

이번:
- PDF를 직접 읽지 않음
- 이미 구조화된 `children.jsonl` 사용
- child chunk 단위로 OpenAI text-embedding-3-small embedding
- ChromaDB 저장
- BM25 keyword index 추가 생성

입력:
- data/retrieval/children.jsonl

출력:
- data/chromadb/
- Chroma collection: complypilot_regulations_v2
- data/retrieval/bm25_index/bm25.pkl
- data/retrieval/debug_index_build/index_build_summary.json

이번 노트북에서는 아직 evidence_retriever node에 연결하지 않는다.
이번 노트북에서는 아직 LangGraph workflow를 수정하지 않는다.

In [1]:
# 기본 import / 환경변수 / 경로 설정
from pathlib import Path
import os
import re
import json
import pickle
import shutil
from pprint import pprint
from typing import List, Dict, Any
from collections import Counter

import pandas as pd
import numpy as np

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

try:
    import chromadb
except ImportError:
    raise ImportError("chromadb가 설치되어 있지 않습니다. %pip install chromadb 를 실행하세요.")

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    raise ImportError("rank_bm25가 설치되어 있지 않습니다. %pip install rank_bm25 를 실행하세요.")


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# .env 로드
env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)

print("env_path:", env_path)
print("env exists:", env_path.exists())
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))


# 경로 설정
RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"
CHROMA_DB_DIR = PROJECT_ROOT / "data" / "chromadb"
BM25_DIR = RETRIEVAL_DIR / "bm25_index"
DEBUG_DIR = RETRIEVAL_DIR / "debug_index_build"

CHILDREN_PATH = RETRIEVAL_DIR / "children.jsonl"
BM25_PATH = BM25_DIR / "bm25.pkl"
SUMMARY_PATH = DEBUG_DIR / "index_build_summary.json"

BM25_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DB_DIR.mkdir(parents=True, exist_ok=True)

# 기존 vectordb_test.ipynb는 complypilot_regulations를 사용했지만,
# 구조화 RAG는 v2 collection으로 분리한다.
COLLECTION_NAME = "complypilot_regulations_v2"

EMBEDDING_PROVIDER = "openai"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CHILDREN_PATH:", CHILDREN_PATH)
print("CHROMA_DB_DIR:", CHROMA_DB_DIR)
print("BM25_PATH:", BM25_PATH)
print("COLLECTION_NAME:", COLLECTION_NAME)
print("EMBEDDING_MODEL_NAME:", EMBEDDING_MODEL_NAME)

c:\Users\USER\Desktop\complypilot-jb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


env_path: c:\Users\USER\Desktop\complypilot-jb\.env
env exists: True
OPENAI_API_KEY exists: True
PROJECT_ROOT: c:\Users\USER\Desktop\complypilot-jb
CHILDREN_PATH: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\children.jsonl
CHROMA_DB_DIR: c:\Users\USER\Desktop\complypilot-jb\data\chromadb
BM25_PATH: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\bm25_index\bm25.pkl
COLLECTION_NAME: complypilot_regulations_v2
EMBEDDING_MODEL_NAME: text-embedding-3-small


In [2]:
# children.jsonl 로드
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """
    JSONL 파일을 읽어 dict 리스트로 반환합니다.

    Args:
        path: JSONL 파일 경로

    Return:
        dict 리스트
    """
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows


children = load_jsonl(CHILDREN_PATH)

print("children row 수:", len(children))
pprint(children[0] if children else None)

assert len(children) > 0, "children.jsonl이 비어 있습니다. 03_build_parent_child_chunks.ipynb를 먼저 확인하세요."

children row 수: 2155
{'article_no': '제1조',
 'article_title': '목적',
 'child_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'child_index': 1,
 'child_status': 'ok',
 'child_text': '이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한\n'
               '사항을 규정함을 목적으로 한다.',
 'chunk_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'doc_code': 'financial_consumer_supervisory_regulation',
 'document_priority': 3,
 'document_type': 'supervisory_regulation',
 'effective_date': '2026.4.2.',
 'has_article_no': True,
 'has_page': True,
 'has_parent_id': True,
 'is_long_child': False,
 'is_short_child': False,
 'item_no': '',
 'keywords': [],
 'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
 'page': 1,
 'page_end': 1,
 'page_start': 1,
 'paragraph_no': '',
 'parent_id': 'financial_consumer_supervisory_regulation__article_1',
 'parent_text': '제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 '
           

In [3]:
# children 기본 품질 확인
df_children = pd.DataFrame(children)

print("children shape:", df_children.shape)

display_cols = [
    "law_name",
    "document_type",
    "article_no",
    "article_title",
    "child_index",
    "split_strategy",
    "page",
    "text_length",
    "risk_tags",
    "keywords",
    "chunk_id",
    "parent_id",
]

display(df_children[display_cols].head(20))

print("document_type 분포")
display(df_children["document_type"].value_counts().reset_index())

print("split_strategy 분포")
display(df_children["split_strategy"].value_counts().reset_index())

if "child_status" in df_children.columns:
    print("child_status 분포")
    display(df_children["child_status"].value_counts().reset_index())

children shape: (2155, 34)


,law_name,document_type,article_no,article_title,child_index,split_strategy,page,text_length,risk_tags,keywords,chunk_id,parent_id
0,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제1조,목적,1,whole_article,1,81,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
1,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,1,paragraph_circled,1,1033,[rate_condition_missing],"[금융투자, 대출, 보험, 상환, 여신, 여신전문, 연, 예금, 예금자보호, 투자]",financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
2,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,2,paragraph_circled,1,1075,[],"[금융투자, 대출, 보험, 상환, 카드, 투자]",financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
3,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,3,paragraph_circled,1,103,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
4,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,4,paragraph_circled,1,130,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
5,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,5,paragraph_circled,1,57,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
6,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,6,paragraph_circled,1,621,[],"[금융투자, 투자]",financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
7,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,7,paragraph_circled,1,91,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
8,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,8,paragraph_circled,1,290,[],"[금융투자, 투자]",financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
9,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,9,paragraph_circled,1,86,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...


document_type 분포


,document_type,count
0,law,1017
1,supervisory_regulation,908
2,enforcement_decree,230


split_strategy 분포


,split_strategy,count
0,paragraph_circled,1669
1,numbered_item,350
2,whole_article,136


child_status 분포


,child_status,count
0,ok,1682
1,short_child_review,472
2,long_child_review,1


In [4]:
# Chroma metadata 변환 함수
def list_to_pipe_string(value: Any) -> str:
    """
    list 값을 Chroma metadata에 넣기 좋은 문자열로 변환합니다.

    Args:
        value: list 또는 기타 값

    Return:
        pipe(|)로 연결된 문자열
    """
    if value is None:
        return ""

    if isinstance(value, list):
        return "|".join(str(x) for x in value)

    return str(value)


def safe_int(value: Any, default: int = 0) -> int:
    """
    값을 int로 안전하게 변환합니다.

    Args:
        value: 변환할 값
        default: 변환 실패 시 기본값

    Return:
        int 값
    """
    try:
        return int(value)
    except Exception:
        return default

def build_chroma_metadata(child: Dict[str, Any]) -> Dict[str, Any]:
    """
    child row에서 ChromaDB에 저장할 metadata를 생성합니다.

    Args:
        child: child chunk row

    Return:
        Chroma metadata dict
    """
    return {
        "chunk_id": str(child.get("chunk_id", "")),
        "parent_id": str(child.get("parent_id", "")),
        "law_name": str(child.get("law_name", "")),
        "document_type": str(child.get("document_type", "")),
        "document_priority": safe_int(child.get("document_priority", 9), 9),

        "article_no": str(child.get("article_no", "")),
        "article_title": str(child.get("article_title", "")),
        "paragraph_no": str(child.get("paragraph_no", "")),
        "item_no": str(child.get("item_no", "")),
        "subitem_no": str(child.get("subitem_no", "")),

        "page": safe_int(child.get("page", 0), 0),
        "page_start": safe_int(child.get("page_start", 0), 0),
        "page_end": safe_int(child.get("page_end", 0), 0),

        "effective_date": str(child.get("effective_date", "")),
        "source_file": str(child.get("source_file", "")),

        "risk_tags": list_to_pipe_string(child.get("risk_tags", [])),
        "product_types": list_to_pipe_string(child.get("product_types", [])),
        "keywords": list_to_pipe_string(child.get("keywords", [])),

        "split_strategy": str(child.get("split_strategy", "")),
        "child_status": str(child.get("child_status", "")),
        "text_length": safe_int(child.get("text_length", 0), 0),
    }

sample_metadata = build_chroma_metadata(children[0])
pprint(sample_metadata)

{'article_no': '제1조',
 'article_title': '목적',
 'child_status': 'ok',
 'chunk_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'document_priority': 3,
 'document_type': 'supervisory_regulation',
 'effective_date': '2026.4.2.',
 'item_no': '',
 'keywords': '',
 'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
 'page': 1,
 'page_end': 1,
 'page_start': 1,
 'paragraph_no': '',
 'parent_id': 'financial_consumer_supervisory_regulation__article_1',
 'product_types': '',
 'risk_tags': '',
 'source_file': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
 'split_strategy': 'whole_article',
 'subitem_no': '',
 'text_length': 81}


In [5]:
# LangChain Document 생성
def build_document_text(child: Dict[str, Any]) -> str:
    """
    Vector DB에 넣을 문서 텍스트를 생성합니다.
    기존 vectordb_test.ipynb처럼 문서명/페이지/조문 정보를 앞에 붙여 검색 안정성을 높입니다.

    Args:
        child: child chunk row

    Return:
        Chroma에 저장할 page_content
    """
    law_name = child.get("law_name", "")
    article_no = child.get("article_no", "")
    article_title = child.get("article_title", "")
    page = child.get("page", child.get("page_start", ""))
    text = child.get("text", "")

    prefix = f"[문서명: {law_name} / 조문: {article_no}({article_title}) / 페이지: {page}]"

    return f"{prefix}\n{text}".strip()


docs = []
ids = []

for child in children:
    chunk_id = str(child.get("chunk_id", "")).strip()
    text = str(child.get("text", "")).strip()

    if not chunk_id or not text:
        continue

    doc = Document(
        page_content=build_document_text(child),
        metadata=build_chroma_metadata(child),
    )

    docs.append(doc)
    ids.append(chunk_id)

print("docs 수:", len(docs))
print("ids 수:", len(ids))

assert len(docs) == len(ids), "docs와 ids 수가 다릅니다."
assert len(set(ids)) == len(ids), "chunk_id 중복이 있습니다."

print("[OK] LangChain Document 생성 완료")
print("샘플 Document:")
print(docs[0].page_content[:700])
pprint(docs[0].metadata)

docs 수: 2155
ids 수: 2155
[OK] LangChain Document 생성 완료
샘플 Document:
[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제1조(목적) / 페이지: 1]
제1조(목적)
이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한
사항을 규정함을 목적으로 한다.
{'article_no': '제1조',
 'article_title': '목적',
 'child_status': 'ok',
 'chunk_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'document_priority': 3,
 'document_type': 'supervisory_regulation',
 'effective_date': '2026.4.2.',
 'item_no': '',
 'keywords': '',
 'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
 'page': 1,
 'page_end': 1,
 'page_start': 1,
 'paragraph_no': '',
 'parent_id': 'financial_consumer_supervisory_regulation__article_1',
 'product_types': '',
 'risk_tags': '',
 'source_file': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
 'split_strategy': 'whole_article',
 'subitem_no': '',
 'text_length': 81}


In [6]:
# OpenAI Embedding 모델 생성
embedding_model = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME
)

test_embedding = embedding_model.embed_query("테스트 문장입니다.")

print("embedding dim:", len(test_embedding))
print("embedding sample:", test_embedding[:5])

assert len(test_embedding) == 1536, "text-embedding-3-small 기본 차원은 보통 1536입니다."

embedding dim: 1536
embedding sample: [-0.004650115966796875, 0.02386474609375, -0.019012451171875, 0.00600433349609375, 0.018768310546875]


In [7]:
# 기존 v2 collection 삭제 후 새로 생성
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))

existing_collections = client.list_collections()
existing_names = [col.name for col in existing_collections]

print("기존 collections:", existing_names)

if COLLECTION_NAME in existing_names:
    print(f"기존 collection 삭제: {COLLECTION_NAME}")
    client.delete_collection(name=COLLECTION_NAME)

# LangChain Chroma wrapper 생성
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DB_DIR),
    collection_metadata={
        "description": "ComplyPilot structured financial regulation evidence chunks",
        "embedding_provider": EMBEDDING_PROVIDER,
        "embedding_model": EMBEDDING_MODEL_NAME,
    },
)

print("VectorStore 생성 완료")
print("COLLECTION_NAME:", COLLECTION_NAME)

기존 collections: []
VectorStore 생성 완료
COLLECTION_NAME: complypilot_regulations_v2


In [8]:
# ChromaDB에 Document batch 추가 - 실제로 Vector DB가 구축
def batch_add_documents_to_chroma(
    vectorstore: Chroma,
    docs: List[Document],
    ids: List[str],
    batch_size: int = 128,
) -> None:
    """
    Chroma vectorstore에 문서를 batch 단위로 추가합니다.

    Args:
        vectorstore: LangChain Chroma vectorstore
        docs: Document 리스트
        ids: chunk_id 리스트
        batch_size: batch size

    Return:
        None
    """
    total = len(docs)

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)

        vectorstore.add_documents(
            documents=docs[start:end],
            ids=ids[start:end],
        )

        print(f"Chroma add 완료: {end}/{total}")


batch_add_documents_to_chroma(
    vectorstore=vectorstore,
    docs=docs,
    ids=ids,
    batch_size=128,
)

print("[OK] ChromaDB add_documents 완료")

Chroma add 완료: 128/2155
Chroma add 완료: 256/2155
Chroma add 완료: 384/2155
Chroma add 완료: 512/2155
Chroma add 완료: 640/2155
Chroma add 완료: 768/2155
Chroma add 완료: 896/2155
Chroma add 완료: 1024/2155
Chroma add 완료: 1152/2155
Chroma add 완료: 1280/2155
Chroma add 완료: 1408/2155
Chroma add 완료: 1536/2155
Chroma add 완료: 1664/2155
Chroma add 완료: 1792/2155
Chroma add 완료: 1920/2155
Chroma add 완료: 2048/2155
Chroma add 완료: 2155/2155
[OK] ChromaDB add_documents 완료


In [16]:
# Chroma collection count 확인
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))
collection = client.get_collection(name=COLLECTION_NAME)

chroma_count = collection.count()

print("collection name:", COLLECTION_NAME)
print("collection count:", chroma_count)
print("expected docs:", len(docs))

assert chroma_count == len(docs), "Chroma collection count가 docs 수와 다릅니다."

print("[OK] ChromaDB Vector Index 생성 완료")

collection name: complypilot_regulations_v2
collection count: 2155
expected docs: 2155
[OK] ChromaDB Vector Index 생성 완료


In [17]:
# Chroma 저장 파일 확인
print("CHROMA_DB_DIR 존재:", CHROMA_DB_DIR.exists())

for path in sorted(CHROMA_DB_DIR.glob("*")):
    print("-", path.name)

CHROMA_DB_DIR 존재: True
- 89071ffb-e2c5-41ab-8db9-02c715d12287
- chroma.sqlite3


In [18]:
# Vector search 함수
def vector_search(
    query: str,
    top_k: int = 5,
    filter_dict: Dict[str, Any] | None = None,
) -> List[Dict[str, Any]]:
    """
    Chroma vector search를 수행합니다.

    Args:
        query: 검색 질의
        top_k: 반환 개수
        filter_dict: Chroma metadata filter

    Return:
        검색 결과 리스트
    """
    if filter_dict:
        docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
            query=query,
            k=top_k,
            filter=filter_dict,
        )
    else:
        docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
            query=query,
            k=top_k,
        )

    rows = []

    for rank, (doc, score) in enumerate(docs_with_scores, start=1):
        metadata = dict(doc.metadata)

        rows.append({
            "rank": rank,
            "chunk_id": metadata.get("chunk_id", f"vector_result_{rank}"),
            "score": float(score),
            "text": doc.page_content,
            **metadata,
        })

    return rows


# Vector search 단독 테스트
test_vector_results = vector_search("금융광고 설명의무 고지", top_k=5)

for row in test_vector_results:
    print("=" * 100)
    print(
        row["rank"],
        row["law_name"],
        row["article_no"],
        row["article_title"],
        "score:",
        round(row["score"], 4),
        "type:",
        row["document_type"],
        "chunk_id:",
        row["chunk_id"][:50],
    )
    print(row["text"][:400])

1 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제18조 광고의 방법 및 절차 score: 0.3937 type: supervisory_regulation chunk_id: financial_consumer_supervisory_regulation__article
[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제18조(광고의 방법 및 절차) / 페이지: 18]
제18조(광고의 방법 및 절차)
영 제19조제1항에서 "금융위원회가 정하여 고시하는 기준"이란 광고에서 글자의 색깔
ㆍ크기 또는 음성의 속도ㆍ크기 등이 해당 금융상품으로 인해 금융소비자가 받을 수 있는 혜택과 불이익을 균형
있게 전달할 것을 말한다.
2 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제19조 광고 시 금지행위 score: 0.3844 type: supervisory_regulation chunk_id: financial_consumer_supervisory_regulation__article
[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제19조(광고 시 금지행위) / 페이지: 18]
제19조(광고 시 금지행위)
① 영 제20조제1항제6호에서 "금융위원회가 정하여 고시하는 행위"란 다음 각 호의 구분
에 따른 행위를 말한다.
1. 금융소비자에 따라 달라질 수 있는 거래조건을 누구에게나 적용될 수 있는 것처럼 오인하게 만드는 행위
2. 보험금 지급사유나 지급시점이 다름에도 불구하고 각각의 보험금이 한꺼번에 지급되는 것처럼 오인하게 만드
는 행위
3. 금융상품에 관한 광고에 연계하여 「보험업법 시행령」 제46조에서 정한 금액을 초과하는 금품을 금융소비자에
제공하는 행위
4. 제17조제3항제1호 각 목의 기준을 충족하는 광고로서 다음 각 목
3 금융소비자 보호에 관한 감독규정(금융위원회고시)(제

In [19]:
# BM25 tokenizer 생성
def tokenize_for_bm25(text: str) -> List[str]:
    """
    BM25 검색용 간단 토큰화를 수행합니다.

    Args:
        text: 입력 텍스트

    Return:
        토큰 리스트
    """
    text = text.lower()
    tokens = re.findall(r"[가-힣A-Za-z0-9]+", text)

    # 너무 짧은 토큰은 제거
    tokens = [token for token in tokens if len(token) >= 2]

    return tokens


sample_query = "누구나 승인 가능한 최저금리 대출 광고의 설명의무와 수수료 고지"
print(tokenize_for_bm25(sample_query))

['누구나', '승인', '가능한', '최저금리', '대출', '광고의', '설명의무와', '수수료', '고지']


In [20]:
# BM25 corpus 생성
bm25_documents = [doc.page_content for doc in docs]
bm25_metadatas = [doc.metadata for doc in docs]
bm25_ids = ids

tokenized_corpus = [tokenize_for_bm25(text) for text in bm25_documents]

print("BM25 corpus size:", len(tokenized_corpus))
print("첫 문서 토큰 샘플:")
print(tokenized_corpus[0][:50])

empty_token_docs = sum(1 for tokens in tokenized_corpus if len(tokens) == 0)
print("empty token docs:", empty_token_docs)

assert len(tokenized_corpus) == len(docs), "BM25 corpus 수가 docs 수와 다릅니다."

print("[OK] BM25 corpus 생성 완료")

BM25 corpus size: 2155
첫 문서 토큰 샘플:
['문서명', '금융소비자', '보호에', '관한', '감독규정', '금융위원회고시', '제2026', '11호', '20260402', '조문', '제1조', '목적', '페이지', '제1조', '목적', '규정은', '금융소비자', '보호에', '관한', '법률', '같은', '시행령에서', '위임하는', '사항과', '시행에', '필요한', '사항을', '규정함을', '목적으로', '한다']
empty token docs: 0
[OK] BM25 corpus 생성 완료


In [21]:
# BM25 index 생성 및 저장
bm25 = BM25Okapi(tokenized_corpus)

bm25_payload = {
    "bm25": bm25,
    "ids": bm25_ids,
    "documents": bm25_documents,
    "metadatas": bm25_metadatas,
    "children": children,
    "tokenizer": "regex_korean_alnum_v1",
    "collection_name": COLLECTION_NAME,
    "embedding_provider": EMBEDDING_PROVIDER,
    "embedding_model": EMBEDDING_MODEL_NAME,
}

with open(BM25_PATH, "wb") as f:
    pickle.dump(bm25_payload, f)

print("BM25 index 저장 완료:", BM25_PATH)
print("파일 크기 MB:", round(BM25_PATH.stat().st_size / (1024 * 1024), 2))

BM25 index 저장 완료: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\bm25_index\bm25.pkl
파일 크기 MB: 11.21


In [22]:
# BM25 index 재로드 검증
with open(BM25_PATH, "rb") as f:
    reloaded_bm25_payload = pickle.load(f)

print("payload keys:", reloaded_bm25_payload.keys())
print("ids count:", len(reloaded_bm25_payload["ids"]))
print("documents count:", len(reloaded_bm25_payload["documents"]))
print("collection_name:", reloaded_bm25_payload["collection_name"])

assert len(reloaded_bm25_payload["ids"]) == len(ids), "BM25 payload ids 수가 다릅니다."
assert len(reloaded_bm25_payload["documents"]) == len(docs), "BM25 payload documents 수가 다릅니다."

print("[OK] BM25 index 재로드 검증 완료")

payload keys: dict_keys(['bm25', 'ids', 'documents', 'metadatas', 'children', 'tokenizer', 'collection_name', 'embedding_provider', 'embedding_model'])
ids count: 2155
documents count: 2155
collection_name: complypilot_regulations_v2
[OK] BM25 index 재로드 검증 완료


In [23]:
# BM25 search 함수
def bm25_search(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """
    BM25 keyword search를 수행합니다.

    Args:
        query: 검색 질의
        top_k: 반환 개수

    Return:
        검색 결과 리스트
    """
    query_tokens = tokenize_for_bm25(query)

    if not query_tokens:
        return []

    scores = bm25.get_scores(query_tokens)
    ranked_indices = np.argsort(scores)[::-1]

    rows = []

    for idx in ranked_indices:
        idx = int(idx)
        score = float(scores[idx])

        # 중요: 실제 lexical match가 없는 0점 결과는 제거
        if score <= 0:
            continue

        metadata = dict(bm25_metadatas[idx])

        rows.append({
            "rank": len(rows) + 1,
            "chunk_id": bm25_ids[idx],
            "score": score,
            "text": bm25_documents[idx],
            **metadata,
        })

        if len(rows) >= top_k:
            break

    return rows


test_bm25_results = bm25_search("수수료 설명의무", top_k=5)

for row in test_bm25_results:
    print("=" * 100)
    print(
        row["rank"],
        row["law_name"],
        row["article_no"],
        row["article_title"],
        "score:",
        round(row["score"], 4),
        "type:",
        row["document_type"],
    )
    print(row["text"][:400])

1 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제13조 설명의무 score: 10.3539 type: enforcement_decree
[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: 제13조(설명의무) / 페이지: 9]
제13조(설명의무)
④ 법 제19조제1항제1호나목4)에서 “대통령령으로 정하는 사항”이란 다음 각 호의 사항(연계투자는 제4호만 해당
한다)을 말한다.
1. 금융소비자가 부담해야 하는 수수료
2. 계약의 해지ㆍ해제
3. 증권의 환매(還買) 및 매매
4. 「온라인투자연계금융업 및 이용자 보호에 관한 법률」 제22조제1항 각 호의 정보
5. 그 밖에 제1호부터 제4호까지의 사항에 준하는 것으로서 금융위원회가 정하여 고시하는 사항
2 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제12조 설명의무 score: 7.7605 type: supervisory_regulation
[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제12조(설명의무) / 페이지: 9]
제12조(설명의무)
① 영 제13조제1항제5호에서 "금융위원회가 정하여 고시하는 사항"이란 별표 3 제1호 각 목의 사
항을 말한다.
3 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제13조 설명의무 score: 7.6391 type: enforcement_decree
[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: 제13조(설명의무) / 페이지: 9]
제13조(설명의무)
⑨ 법 제19조제2항 본문에서 “대통령령으로 정하는 방법”이란 제11조제2항에 따른 방법을 말한다.<신설 2022. 12.
8.>
4 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제12조 설명의무 score: 7

In [24]:
# Seed query smoke test
SEED_QUERIES = [
    "누구나 승인",
    "최저금리",
    "수수료",
    "원금보장",
    "확정수익",
    "설명의무",
    "부당권유",
    "금리",
    "이자율",
    "광고 오인",
]

for query in SEED_QUERIES:
    print("=" * 120)
    print("QUERY:", query)

    print("\n[Vector Top 3]")
    vector_rows = vector_search(query, top_k=3)
    for row in vector_rows:
        print(
            f"- {row['law_name']} {row['article_no']}({row['article_title']}) "
            f"score={row['score']:.4f} type={row['document_type']}"
        )

    print("\n[BM25 Top 3]")
    bm25_rows = bm25_search(query, top_k=3)
    for row in bm25_rows:
        print(
            f"- {row['law_name']} {row['article_no']}({row['article_title']}) "
            f"score={row['score']:.4f} type={row['document_type']}"
        )

QUERY: 누구나 승인

[Vector Top 3]
- 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401) 제51조(비금융자회사 출자승인) score=0.0927 type=supervisory_regulation
- 예금자보호법(법률)(제21065호)(20260102) 제35조의4(개산지급금 지급의 승인) score=0.0696 type=law
- 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401) 제51조(비금융자회사 출자승인) score=0.0688 type=supervisory_regulation

[BM25 Top 3]
- 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401) 제9조(자본금 감소의 승인) score=8.2622 type=supervisory_regulation
- 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401) 제9조(자본금 감소의 승인) score=8.2622 type=supervisory_regulation
- 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401) 제9조(자본금 감소의 승인) score=7.8938 type=supervisory_regulation
QUERY: 최저금리

[Vector Top 3]
- 여신전문금융업법(법률)(제21065호)(20251001) 제50조의13(금리인하 요구) score=0.0735 type=law
- 여신전문금융업법(법률)(제21065호)(20251001) 제50조의13(금리인하 요구) score=0.0564 type=law
- 여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506) 제24조의3(길거리의범위) score=0.0403 type=supervisory_regulation

[BM25 Top 3]
QUERY: 수수료

[Vector Top 3]
- 여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506) 제25조의6(우대수수료율) score=0.1379 

In [25]:
# 간단 Hybrid merge 테스트
def reciprocal_rank_fusion(
    result_lists: List[List[Dict[str, Any]]],
    k: int = 60,
) -> List[Dict[str, Any]]:
    """
    여러 검색 결과를 RRF 방식으로 병합합니다.

    Args:
        result_lists: 검색 결과 리스트들의 리스트
        k: RRF 보정 상수

    Return:
        병합된 검색 결과 리스트
    """
    scores = {}
    items = {}
    methods = {}

    for result_list in result_lists:
        for rank, item in enumerate(result_list, start=1):
            chunk_id = item.get("chunk_id", "")

            if not chunk_id:
                continue

            scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)

            if chunk_id not in items:
                items[chunk_id] = dict(item)

            methods.setdefault(chunk_id, set()).add(
                item.get("retrieval_method", "unknown")
            )

    merged = []

    for chunk_id, item in items.items():
        row = dict(item)
        row["rrf_score"] = scores[chunk_id]
        row["retrieval_method"] = "+".join(sorted(methods.get(chunk_id, [])))
        merged.append(row)

    merged = sorted(merged, key=lambda x: x["rrf_score"], reverse=True)

    return merged


def hybrid_smoke_search(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """
    vector와 BM25 결과를 간단히 병합합니다.

    Args:
        query: 검색 질의
        top_k: 반환 개수

    Return:
        병합 검색 결과
    """
    vector_rows = vector_search(query, top_k=10)
    bm25_rows = bm25_search(query, top_k=10)

    for row in vector_rows:
        row["retrieval_method"] = "vector"

    for row in bm25_rows:
        row["retrieval_method"] = "bm25"

    merged = reciprocal_rank_fusion([vector_rows, bm25_rows])

    deduped = []
    seen_parent_ids = set()

    for row in merged:
        parent_id = row.get("parent_id", "")

        if parent_id in seen_parent_ids:
            continue

        seen_parent_ids.add(parent_id)
        deduped.append(row)

        if len(deduped) >= top_k:
            break

    return deduped


hybrid_results = hybrid_smoke_search("수수료 고지 설명의무", top_k=5)

for row in hybrid_results:
    print("=" * 100)
    print(
        row["law_name"],
        row["article_no"],
        row["article_title"],
        "rrf_score:",
        round(row["rrf_score"], 5),
        "type:",
        row["document_type"],
        "method:",
        row.get("retrieval_method"),
    )
    print(row["text"][:400])

금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제12조 설명의무 rrf_score: 0.03055 type: supervisory_regulation method: bm25+vector
[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제12조(설명의무) / 페이지: 9]
제12조(설명의무)
④ 영 제13조제4항제5호에서 "금융위원회가 정하여 고시하는 사항"이란 별표 3 제2호 각 목의 구분에 따른 사항
을 말한다.
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제8조 등록수수료 rrf_score: 0.01639 type: supervisory_regulation method: vector
[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제8조(등록수수료) / 페이지: 6]
제8조(등록수수료)
영 제9조에서 "금융위원회가 정하여 고시하는 수수료"란 다음 각 호의 구분에 따른 금액을 말한
다.
금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제13조 설명의무 rrf_score: 0.01639 type: enforcement_decree method: bm25
[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: 제13조(설명의무) / 페이지: 9]
제13조(설명의무)
④ 법 제19조제1항제1호나목4)에서 “대통령령으로 정하는 사항”이란 다음 각 호의 사항(연계투자는 제4호만 해당
한다)을 말한다.
1. 금융소비자가 부담해야 하는 수수료
2. 계약의 해지ㆍ해제
3. 증권의 환매(還買) 및 매매
4. 「온라인투자연계금융업 및 이용자 보호에 관한 법률」 제22조제1항 각 호의 정보
5. 그 밖에 제1호부터 제4호까지의 사항에 준하는 것으로서 금융위원회가 정하여 고시하는 사항
예금자보호법(법률)(제

In [26]:
# Report-safe evidence format 미리보기
def to_report_evidence(row: Dict[str, Any]) -> Dict[str, Any]:
    """
    검색 결과 row를 report/UI에 안전하게 노출할 evidence format으로 변환합니다.

    Args:
        row: 검색 결과 row

    Return:
        report-safe evidence dict
    """
    doc_title = f"{row.get('law_name', '')} {row.get('article_no', '')}({row.get('article_title', '')})"
    snippet = row.get("text", "").replace("\n", " ").strip()

    return {
        "doc_title": doc_title,
        "page": row.get("page", row.get("page_start", "")),
        "snippet": snippet[:500],
        "score": round(float(row.get("rrf_score", row.get("score", 0.0))), 5),
        "retrieval_method": row.get("retrieval_method", "hybrid_smoke"),
    }


report_evidences = [to_report_evidence(row) for row in hybrid_results]

pprint(report_evidences)

[{'doc_title': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제12조(설명의무)',
  'page': 9,
  'retrieval_method': 'bm25+vector',
  'score': 0.03055,
  'snippet': '[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: '
             '제12조(설명의무) / 페이지: 9] 제12조(설명의무) ④ 영 제13조제4항제5호에서 "금융위원회가 정하여 '
             '고시하는 사항"이란 별표 3 제2호 각 목의 구분에 따른 사항 을 말한다.'},
 {'doc_title': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제8조(등록수수료)',
  'page': 6,
  'retrieval_method': 'vector',
  'score': 0.01639,
  'snippet': '[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: '
             '제8조(등록수수료) / 페이지: 6] 제8조(등록수수료) 영 제9조에서 "금융위원회가 정하여 고시하는 수수료"란 '
             '다음 각 호의 구분에 따른 금액을 말한 다.'},
 {'doc_title': '금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제13조(설명의무)',
  'page': 9,
  'retrieval_method': 'bm25',
  'score': 0.01639,
  'snippet': '[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: '
             '제13조(설명의무) / 페이지: 9] 제13조(설명의무) ④ 법 제19조제1항제1호나목4)에서 “대통령령으로 정하는 '
           

In [27]:
# index build summary 저장
index_summary = {
    "collection_name": COLLECTION_NAME,
    "embedding_provider": EMBEDDING_PROVIDER,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "children_count": len(children),
    "docs_count": len(docs),
    "chroma_count": chroma_count,
    "chroma_dir": str(CHROMA_DB_DIR),
    "bm25_path": str(BM25_PATH),
    "bm25_exists": BM25_PATH.exists(),
    "bm25_file_size_mb": round(BM25_PATH.stat().st_size / (1024 * 1024), 2) if BM25_PATH.exists() else 0,
    "document_type_counts": df_children["document_type"].value_counts().to_dict(),
    "split_strategy_counts": df_children["split_strategy"].value_counts().to_dict(),
    "seed_queries": SEED_QUERIES,
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(index_summary, f, ensure_ascii=False, indent=2)

print("index build summary 저장:", SUMMARY_PATH)
pprint(index_summary)

index build summary 저장: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_index_build\index_build_summary.json
{'bm25_exists': True,
 'bm25_file_size_mb': 11.21,
 'bm25_path': 'c:\\Users\\USER\\Desktop\\complypilot-jb\\data\\retrieval\\bm25_index\\bm25.pkl',
 'children_count': 2155,
 'chroma_count': 2155,
 'chroma_dir': 'c:\\Users\\USER\\Desktop\\complypilot-jb\\data\\chromadb',
 'collection_name': 'complypilot_regulations_v2',
 'docs_count': 2155,
 'document_type_counts': {'enforcement_decree': 230,
                          'law': 1017,
                          'supervisory_regulation': 908},
 'embedding_model': 'text-embedding-3-small',
 'embedding_provider': 'openai',
 'seed_queries': ['누구나 승인',
                  '최저금리',
                  '수수료',
                  '원금보장',
                  '확정수익',
                  '설명의무',
                  '부당권유',
                  '금리',
                  '이자율',
                  '광고 오인'],
 'split_strategy_counts': {'numbered_item': 350,
 

In [28]:
# 최종 체크
print("=" * 100)
print("04_build_vector_bm25_index 최종 체크")
print("=" * 100)

vector_check_rows = vector_search("설명의무", top_k=3)
bm25_zero_check_rows = bm25_search("존재하지않는이상한질의어", top_k=3)

checks = {
    "children_count_gt_0": len(children) > 0,
    "docs_count_gt_0": len(docs) > 0,
    "chroma_dir_exists": CHROMA_DB_DIR.exists(),
    "collection_name_ok": COLLECTION_NAME == "complypilot_regulations_v2",
    "chroma_count_match": chroma_count == len(docs),
    "bm25_path_exists": BM25_PATH.exists(),
    "bm25_documents_count_match": len(reloaded_bm25_payload["documents"]) == len(docs),
    "vector_search_returns": len(vector_check_rows) > 0,
    "bm25_search_returns": len(bm25_search("수수료", top_k=3)) > 0,
    "hybrid_smoke_returns": len(hybrid_smoke_search("광고 설명의무", top_k=3)) > 0,

    # 추가 체크
    "vector_chunk_id_is_real": all(
        not row["chunk_id"].startswith("vector_result_")
        for row in vector_check_rows
    ),
    "bm25_zero_score_filtered": len(bm25_zero_check_rows) == 0,
}

pprint(checks)

if all(checks.values()):
    print("[OK] ChromaDB Vector Index + BM25 Index 생성 완료")
else:
    print("[WARN] 일부 체크가 실패했습니다. 위 결과를 보고 보완이 필요합니다.")

04_build_vector_bm25_index 최종 체크
{'bm25_documents_count_match': True,
 'bm25_path_exists': True,
 'bm25_search_returns': True,
 'bm25_zero_score_filtered': True,
 'children_count_gt_0': True,
 'chroma_count_match': True,
 'chroma_dir_exists': True,
 'collection_name_ok': True,
 'docs_count_gt_0': True,
 'hybrid_smoke_returns': True,
 'vector_chunk_id_is_real': True,
 'vector_search_returns': True}
[OK] ChromaDB Vector Index + BM25 Index 생성 완료
